In [ ]:
# Imports and requests

import requests
import pandas as pd


BASE_URL = "https://api.ratings.food.gov.uk"
HEADERS = {"x-api-version": "2", "Accept": "application/json"}

pd.set_option("display.max_columns", None)  # show every column

In [ ]:
# Call to the Authorities Endpoint
response = requests.get(f"{BASE_URL}/Authorities", headers=HEADERS, timeout=30)

print("Status code:", response.status_code)
data = response.json()
print("Top-level keys:", list(data.keys()))
print("Number of authorities:", len(data["authorities"]))

Status code: 200
Top-level keys: ['authorities', 'meta', 'links']
Number of authorities: 363


In [ ]:
# Creating a Dataframe

authorities = pd.DataFrame(data["authorities"])

print(authorities.columns.tolist())
print()
print(authorities["RegionName"].value_counts())

['LocalAuthorityId', 'LocalAuthorityIdCode', 'Name', 'FriendlyName', 'Url', 'SchemeUrl', 'Email', 'RegionName', 'FileName', 'FileNameWelsh', 'EstablishmentCount', 'CreationDate', 'LastPublishedDate', 'SchemeType', 'links']

RegionName
South East                  64
East Counties               45
East Midlands               35
North West                  35
London                      33
Scotland                    32
West Midlands               30
South West                  27
Wales                       22
Yorkshire and Humberside    16
North East                  13
Northern Ireland            11
Name: count, dtype: int64


In [4]:
# The London Authorities
london = authorities.loc[
    authorities["RegionName"] == "London",
    ["LocalAuthorityId", "Name", "EstablishmentCount"],
].sort_values("EstablishmentCount", ascending=False)

print(london.to_string(index=False))
print("\nTotal expected London businesses:", london["EstablishmentCount"].sum())

 LocalAuthorityId                       Name  EstablishmentCount
              120                Westminster                5748
               93                     Camden                4271
               96                     Ealing                3813
              115                  Southwark                3310
              117              Tower Hamlets                3187
               94                    Croydon                3119
              119                 Wandsworth                2921
               89                     Barnet                2854
              109                    Lambeth                2669
              112                     Newham                2652
              106                  Islington                2616
               91                      Brent                2500
              110                   Lewisham                2484
               92                    Bromley                2479
               99        

In [5]:
# One page of Business from Westminster

westminster_id = london.loc[london["Name"] == "Westminster", "LocalAuthorityId"].iloc[0]

params = {"localAuthorityId": westminster_id, "pageNumber": 1, "pageSize": 5}
response = requests.get(f"{BASE_URL}/Establishments", headers=HEADERS, params=params, timeout=30)

page = response.json()
print("Status code:", response.status_code)
print("Top-level keys:", list(page.keys()))
print("Meta:", page["meta"])

Status code: 200
Top-level keys: ['establishments', 'meta', 'links']
Meta: {'dataSource': 'ElasticSearch', 'extractDate': '2026-09-22T20:30:40.0732918+01:00', 'itemCount': 5, 'returncode': 'OK', 'totalCount': 5748, 'totalPages': 1150, 'pageSize': 5, 'pageNumber': 1}


In [6]:
# One full Business record

import json

print(json.dumps(page["establishments"][0], indent=2))

{
  "AddressLine1": "",
  "AddressLine2": "",
  "AddressLine3": "",
  "AddressLine4": "",
  "BusinessName": "(ki:ts)",
  "BusinessType": "Retailers - other",
  "BusinessTypeID": 4613,
  "ChangesByServerID": 0,
  "Distance": null,
  "FHRSID": 1916465,
  "LocalAuthorityBusinessID": "LKNM9H-V40BYF-WAVKXY",
  "LocalAuthorityCode": "533",
  "LocalAuthorityEmailAddress": "foodsafety@westminster.gov.uk",
  "LocalAuthorityName": "Westminster",
  "LocalAuthorityWebSite": "http://www.westminster.gov.uk/",
  "NewRatingPending": false,
  "Phone": "",
  "PostCode": "W1U ",
  "RatingDate": "2026-03-04T00:00:00",
  "RatingKey": "fhrs_5_en-gb",
  "RatingValue": "5",
  "RightToReply": "",
  "SchemeType": "FHRS",
  "geocode": {
    "longitude": null,
    "latitude": null
  },
  "scores": {
    "Hygiene": 0,
    "Structural": 0,
    "ConfidenceInManagement": 0
  }
}


## API exploration: findings

### Size of the data
- London has **33 local authorities** (`RegionName == "London"`).
- The Authorities endpoint says London should have **81,572 businesses** in total
  (`EstablishmentCount`). Westminster is the largest (5,748), Sutton the smallest (1,318).
- We'll download at **1,000 businesses per page**, which is roughly 90 requests for all of London.
- Every request needs the header `x-api-version: 2`.
- The `meta` section of each response gives `totalCount` and `totalPages`, which the
  script uses to know when to stop.

### Field groups
**Known before inspection (candidate features)**
- `BusinessType`, `BusinessTypeID`
- `BusinessName` (to spot chains vs independents)
- Address fields, `PostCode`, `geocode.latitude`, `geocode.longitude`
- `LocalAuthorityName`, `LocalAuthorityCode`

**Only known after inspection (never use as features)**
- `RatingValue`: the source of our target (2 or below = fail)
- `RatingKey`: contains the rating inside the text (e.g. `fhrs_5_en-gb`)
- `scores.Hygiene`, `scores.Structural`, `scores.ConfidenceInManagement`:
  the rating is calculated from these (penalty points, 0 = best)
- `NewRatingPending`, `RightToReply`: only exist because an inspection happened
- `RatingDate`: the date of the same inspection that gave the rating. Councils also
  re-inspect poor businesses sooner, so a recent date hints at a bad rating.
  Use it for filtering only, never as a feature.

**Admin / IDs (drop for modelling)**
- Keep `FHRSID` as the unique ID.
- Drop `LocalAuthorityBusinessID`, `ChangesByServerID`, `Distance`, `Phone`,
  `LocalAuthorityEmailAddress`, `LocalAuthorityWebSite`, `SchemeType`.

### Data quality issues spotted
- Some businesses have **empty address lines and a partial postcode** (e.g. `"W1U "` with
  a trailing space), probably because the business asked for its address to be withheld
  (home caterers, online sellers).
- Some businesses have **null latitude and longitude**, so they can't be mapped.
  We need a plan for these during cleaning.

### Key principle
Only use information an inspector would have **before** visiting. Anything produced
by the inspection itself is leakage and would make the model look great but be useless.
